<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/2%20-%20Regress%C3%A3o%20Linear/2.6_regressao_linear_salarios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2.6 — Regressão linear com dados reais: salários (treino × validação)

## Objetivos

- Aplicar o pipeline do 2.5 (`nn.Linear`, `nn.MSELoss`, `optim.SGD`) a um dataset real: salário em função dos anos de experiência.
- Separar os dados em treino e validação (70/30) e acompanhar a função de custo (loss) nos dois conjuntos.
- Organizar o treino em funções `train_epoch` e `validate_epoch`, com `model.train()`, `model.eval()` e `torch.no_grad()`.
- Usar o bias da própria camada (`nn.Linear(1, 1)`) no lugar da coluna de uns.

O problema é o exemplo dos slides: dado o tempo de experiência de um profissional, prever o seu salário. A entrada $x$ é `YearsExperience` (anos de experiência) e a saída $y$ é `Salary` (salário anual; a fonte não informa a moeda). O modelo é a mesma reta dos notebooks anteriores:

$$\hat{y} = w_0 + w_1 x$$

com $w_0$ = bias (intercepto) e $w_1$ = peso. Nos slides: $h_\theta(x) = \theta_0 + \theta_1 x$, com $w_0 = \theta_0$ e $w_1 = \theta_1$.

O que muda em relação ao 2.5: os dados são reais e poucos (30 profissionais), parte deles é separada para validação, e o bias passa a ser um parâmetro da camada.

## Importação das bibliotecas

In [ ]:
import io
import pathlib
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn, optim

torch.manual_seed(1234)
np.random.seed(1234)

## Dados

### Download

Fonte: [Salary Dataset - Simple linear regression](https://www.kaggle.com/datasets/abhishek14398/salary-dataset-simple-linear-regression) (Kaggle). O endpoint público de download do Kaggle entrega o `.zip` sem login. O arquivo `Salary_dataset.csv` tem uma coluna de índice sem nome, `YearsExperience` e `Salary`.

Ele é uma cópia do clássico `Salary_Data.csv`, com 0,1 ano e 1 unidade de salário somados a todas as linhas. Se o Kaggle estiver indisponível, a célula baixa o `Salary_Data.csv` de um repositório público no GitHub. Depois da padronização (abaixo), os dois arquivos dão os mesmos valores, porque somar uma constante não muda a padronização.

O arquivo vai para a pasta `./data`; se já existir, não é baixado de novo.

In [ ]:
DATA_DIR = pathlib.Path('data')
DATA_DIR.mkdir(exist_ok=True)

KAGGLE_URL = 'https://www.kaggle.com/api/v1/datasets/download/abhishek14398/salary-dataset-simple-linear-regression'
MIRROR_URL = 'https://raw.githubusercontent.com/krishnaik06/simple-Linear-Regression/master/Salary_Data.csv'

csv_path = DATA_DIR / 'Salary_dataset.csv'
if not csv_path.exists():
    try:
        with urllib.request.urlopen(KAGGLE_URL, timeout=30) as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extract('Salary_dataset.csv', DATA_DIR)
        print('baixado do Kaggle:', csv_path)
    except Exception as e:
        print('Kaggle indisponível (', e, '); usando a cópia do GitHub')
        csv_path = DATA_DIR / 'Salary_Data.csv'
        urllib.request.urlretrieve(MIRROR_URL, csv_path)
print('arquivo:', csv_path)

### Leitura

In [ ]:
df = pd.read_csv(csv_path)
df = df[['YearsExperience', 'Salary']]   # descarta a coluna de índice, se houver
print('shape:', df.shape)
df.head()

In [ ]:
plt.scatter(df['YearsExperience'], df['Salary'])
plt.xlabel('anos de experiência')
plt.ylabel('salário anual')
plt.title('Salary Data')
plt.show()

### Treino × validação

Os parâmetros são ajustados só com o conjunto de treino. O conjunto de validação fica de fora do treino e serve para medir o custo em dados que o modelo não viu: um custo baixo no treino não garante um custo baixo em dados novos.

`train_test_split(..., test_size=0.3, random_state=42)` sorteia 30% das linhas para validação: 21 amostras de treino e 9 de validação. `random_state` fixa o sorteio, para que a divisão seja sempre a mesma. Com só 30 amostras, não separamos um terceiro conjunto (teste).

In [ ]:
x_raw = df[['YearsExperience']].to_numpy()   # (30, 1)
y_raw = df[['Salary']].to_numpy()            # (30, 1)

x_train_raw, x_val_raw, y_train_raw, y_val_raw = train_test_split(
    x_raw, y_raw, test_size=0.3, random_state=42)
print('treino:', x_train_raw.shape, ' validação:', x_val_raw.shape)

### Normalização

Anos de experiência ficam entre 1 e 11; salários, entre ~37 mil e ~122 mil. Com escalas tão diferentes, o gradiente descendente precisaria de uma taxa de aprendizado minúscula. Padronizamos cada variável (z-score):

$$z = \frac{x - \mu}{\sigma}$$

em que $\mu$ e $\sigma$ são a média e o desvio padrão. O `StandardScaler` do scikit-learn faz essa conta: `fit` calcula $\mu$ e $\sigma$, `transform` aplica a fórmula e `inverse_transform` desfaz.

$\mu$ e $\sigma$ são calculados **só no treino** e depois aplicados à validação. Se fossem calculados com todos os dados, informação da validação entraria no treino (vazamento de dados), e a validação deixaria de representar dados novos.

In [ ]:
x_scaler = StandardScaler().fit(x_train_raw)
y_scaler = StandardScaler().fit(y_train_raw)

# tensores float32 (sem requires_grad: são dados, não parâmetros)
x_train = torch.from_numpy(x_scaler.transform(x_train_raw)).float()
y_train = torch.from_numpy(y_scaler.transform(y_train_raw)).float()
x_val = torch.from_numpy(x_scaler.transform(x_val_raw)).float()
y_val = torch.from_numpy(y_scaler.transform(y_val_raw)).float()

print('x_train:', x_train.shape, ' y_train:', y_train.shape)
print('x_val:  ', x_val.shape, ' y_val:  ', y_val.shape)

## Modelo

`nn.Linear(1, 1)` tem `bias=True` por padrão e calcula $\hat{y} = w_1 x + w_0$:

- `model.weight`, shape (1, 1), é o peso $w_1$;
- `model.bias`, shape (1,), é o bias $w_0$.

São os mesmos 2 parâmetros dos notebooks anteriores, mas agora não é preciso a coluna de uns (`x_train_bias`): a camada soma o bias. Usamos a inicialização padrão do `nn.Linear`.

In [ ]:
model = nn.Linear(1, 1)
print('weight (w1):', model.weight)
print('bias (w0):  ', model.bias)

### Testando o predict da rede

In [ ]:
model(torch.ones(5, 1))

In [ ]:
model(x_train[:3])

## Treinamento

### Função de custo e otimizador

Os mesmos do 2.5.

In [ ]:
lr = 0.3
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)   # atualiza weight e bias

### Funções de treino e de validação

O código de uma época vira duas funções:

- `train_epoch` faz um passo de gradiente descendente com todo o conjunto de treino (as 21 amostras de uma vez; aqui uma época é uma iteração. Mini-batches aparecem no 3.3) e devolve o custo;
- `validate_epoch` só calcula o custo na validação, sem atualizar nada.

Duas construções novas:

- `model.train()` e `model.eval()` colocam o modelo em modo de treino ou de avaliação. Para `nn.Linear` não muda nada; muda para camadas como dropout e batch normalization, que se comportam de forma diferente no treino (módulo 4). Usar os dois desde já é o padrão do PyTorch.
- `with torch.no_grad():` desliga o registro do grafo do autograd. Na validação não há `backward()`, então o grafo seria memória e tempo desperdiçados.

In [ ]:
def train_epoch(model, optimizer, criterion, x_train, y_train):
    model.train()            # modo de treino
    optimizer.zero_grad()

    # forward
    y_pred = model(x_train)
    loss = criterion(y_pred, y_train)

    # backward e otimizador
    loss.backward()
    optimizer.step()

    return loss.item()       # custo como número Python

In [ ]:
def validate_epoch(model, criterion, x_val, y_val):
    model.eval()             # modo de avaliação
    with torch.no_grad():    # sem grafo do autograd: não haverá backward
        y_pred = model(x_val)
        loss = criterion(y_pred, y_val)
    return loss.item()

### Laço de treinamento

A cada época: um passo de treino, a medida na validação e o registro dos parâmetros. Observe que `train_loss` é medido no forward, antes do `step()` daquela época, e `val_loss` depois dele.

In [ ]:
n_epochs = 120
w0_list = []   # bias
w1_list = []   # peso
train_losses = []
val_losses = []

for epoch in range(n_epochs):
    w0_list.append(model.bias.item())
    w1_list.append(model.weight.item())

    train_loss = train_epoch(model, optimizer, criterion, x_train, y_train)
    train_losses.append(train_loss)

    val_loss = validate_epoch(model, criterion, x_val, y_val)
    val_losses.append(val_loss)

    if (epoch + 1) % 20 == 0:
        print(f'Epoch[{epoch + 1}/{n_epochs}], Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')

## Avaliação

### Curvas de custo de treino e de validação

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(n_epochs), train_losses, label='treino')
plt.plot(range(n_epochs), val_losses, label='validação')
plt.xlabel('época')
plt.ylabel('função de custo (MSE, dados padronizados)')
plt.title('Custo de treino e de validação')
plt.legend()
plt.grid(True)
plt.show()

### Reta ajustada

A reta é calculada nos dados padronizados e levada de volta à escala original com `inverse_transform`, para ler os eixos em anos e em salário.

In [ ]:
x_grid_raw = np.linspace(x_raw.min(), x_raw.max(), 100).reshape(-1, 1)
x_grid = torch.from_numpy(x_scaler.transform(x_grid_raw)).float()

model.eval()
with torch.no_grad():
    y_grid = model(x_grid)
y_grid_raw = y_scaler.inverse_transform(y_grid.numpy())

plt.figure(figsize=(10, 6))
plt.plot(x_train_raw, y_train_raw, 'ro', label='treino')
plt.plot(x_val_raw, y_val_raw, 'go', label='validação')
plt.plot(x_grid_raw, y_grid_raw, 'k-', label='reta ajustada')
plt.xlabel('anos de experiência')
plt.ylabel('salário anual')
plt.title('Regressão linear: treino e validação')
plt.legend()
plt.grid(True)
plt.show()

### Parâmetros treinados (na escala padronizada)

In [ ]:
print('w0 (bias):', model.bias.item())
print('w1 (peso):', model.weight.item())
print(f'MSE final  treino: {train_losses[-1]:.4f}   validação: {val_losses[-1]:.4f}')

## Exercícios

### Exercício 1 — Erro em unidades de salário

O MSE acima está na escala padronizada, difícil de interpretar. Calcule a raiz do erro quadrático médio (RMSE) na validação **em unidades de salário**:

$$\text{RMSE} = \sqrt{\frac{1}{M_{val}}\sum_{i=1}^{M_{val}}\left(\hat{y}^{(i)} - y^{(i)}\right)^2}$$

em que $M_{val} = 9$ é o número de amostras de validação. Use `y_scaler.inverse_transform` para levar as previsões de volta à escala original e compare com `y_val_raw`. Guarde o resultado em `rmse_val` e compare com o salário médio.

In [ ]:
# TODO: escreva sua solução aqui
print(f'RMSE de validação: {rmse_val:,.0f}')
print(f'salário médio:     {y_raw.mean():,.0f}')

**Resposta:**

*Escreva sua resposta aqui.*

### Exercício 2 — Previsão para um caso novo

Qual salário o modelo prevê para uma pessoa com 5 anos de experiência? Lembre-se de padronizar a entrada com `x_scaler` antes de passá-la ao modelo e de desfazer a padronização da saída. Guarde o valor (um número) em `salario_5`.

In [ ]:
anos = np.array([[5.0]])
# TODO: escreva sua solução aqui
print(f'salário previsto para {anos[0, 0]:.0f} anos: {salario_5:,.0f}')

### Exercício 3 — Taxa de aprendizado

Antes de executar, preveja: o que acontece com as curvas de validação se a taxa de aprendizado for 0,01? E 1,1?

Para cada `lr` em `[0.01, 0.1, 0.3, 1.1]`, crie um modelo novo (`torch.manual_seed(1234)` antes de `nn.Linear(1, 1)`, para todos partirem do mesmo ponto) e um otimizador novo, treine por `n_epochs` épocas com `train_epoch`/`validate_epoch` e plote as curvas de custo de validação no mesmo gráfico (use `plt.yscale('log')`). Qual valor você escolheria e por quê?

In [ ]:
lr_list = [0.01, 0.1, 0.3, 1.1]
plt.figure(figsize=(10, 6))
# TODO: escreva sua solução aqui
plt.yscale('log')
plt.xlabel('época')
plt.ylabel('custo de validação (MSE, escala log)')
plt.legend()
plt.grid(True)
plt.show()

**Resposta:**

*Escreva sua resposta aqui.*

### Exercício 4 — Trajetória dos parâmetros

As listas `w0_list` (bias) e `w1_list` (peso) foram preenchidas no laço de treinamento. Faça o gráfico da trajetória de $(w_0, w_1)$ com setas, como nos notebooks 2.4 e 2.5, e marque a solução analítica (equação normal com uma coluna de uns, calculada com os dados de **treino**). Dica: como os eixos têm escalas bem diferentes, `plt.arrow` deforma as pontas das setas; `plt.annotate('', xy=(w0, w1), xytext=(w0_old, w1_old), arrowprops=dict(arrowstyle='->'))` desenha a seta sem deformação. Por que o $w_0$ ótimo é praticamente zero?

In [ ]:
# TODO: escreva sua solução aqui


**Resposta:**

*Escreva sua resposta aqui.*

## Aprendizados

- Num dataset real, os parâmetros são ajustados só com o treino; a validação mede o custo em dados que o modelo não viu.
- A normalização também é ajustada só no treino (`fit`) e depois aplicada à validação (`transform`), para evitar vazamento de dados; `inverse_transform` traz as previsões de volta para a unidade original.
- `train_epoch`/`validate_epoch` separam o passo de treino (com `backward()` e `step()`) da avaliação (`model.eval()` e `torch.no_grad()`).
- `nn.Linear(1, 1)` tem o bias embutido: `model.bias` é $w_0$ e `model.weight` é $w_1$.
- A taxa de aprendizado decide entre convergência lenta, rápida ou divergência; a curva de validação mostra o efeito.